In [1]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.abspath('../Scripts'))
sys.path.insert(0, os.path.expanduser('~/CDD_Vault_API/python'))  # CDD Vault API (get_df)

/home/gtamo/MS_ML


In [2]:
%load_ext autoreload
%autoreload 2

import re
import os
import pandas as pd
import numpy as np
import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import csv
import pickle

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem,rdFMCS
# import prolif as plf
from glob import glob
import meeko
import subprocess as sub
# from vina import Vina
import time
from tqdm.auto import tqdm  # auto -> notebook/VSCode widget
tqdm.pandas()
import importlib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, silhouette_score, davies_bouldin_score, calinski_harabasz_score
from xgboost import XGBClassifier,XGBRegressor
from openTSNE import TSNE as oTSNE        # pip install openTSNE
import seaborn as sns
from scipy import stats
import pyarrow.dataset as pads
import csv, contextlib, threading, joblib
import joblib
from joblib import Parallel, delayed
from datetime import date
import ipywidgets as w
from IPython.display import clear_output, display

# user defined modules
import Rdkit_tools as rdkit_tools
importlib.reload(rdkit_tools)
import Molecule as M
import ML_Reg as ML_Reg
import ML_Class as ML_Class
from MolViz3D import MolViz3D
import Statistics_tools as stats_tools
import python.functions as fn
from get_library import get_df   # CDD Vault collection export
from python.MS_build_ML import PARAMS, DATA, OUTPUT   # CLI classes reused here (%autoreload picks up edits)
# from tdc.multi_pred import DTI

In [3]:
## params — every YAML key becomes an attribute (params.MS_PATH); Dropbox readers + MLTrail vault bound here
CONFIG = 'config/config.yaml'

params = PARAMS(CONFIG)
params.load_params()
params.setup_dropbox()    # params.dbx / params.dbx_glob per DROPBOX_BACKEND
params.load_registry()    # params.registry (shared MLTrail vault)

## transition shim — keeps the bare YAML names (SERAC_C, MS_PATH, ...) working in the cells below
globals().update({k: v for k, v in params.__dict__.items() if k.isupper()})
registry = params.registry
_dbx = _fbx_open = params.dbx
_fbx_glob = params.dbx_glob

print('> chemlib overwrite:', params.CHEMLIB_OVERWRITE, '| dfraw overwrite:', params.DFRAW_OVERWRITE)
print('> features:', params.FEATURES_TYPE)

> loaded 71 params from config/config.yaml
> Dropbox backend: rclone
> chemlib overwrite: True | dfraw overwrite: False
> features: H237


## 0. Imports

In [4]:
%%time
## data — library + proteomics + chemistry features (see python/MS_build_ML.py:DATA)
data = DATA()
data.load_chemical_lib_df(params)                                  # serac_df + validated / devalidated targets
## cm2rm now comes from config CM2RM_PARTS — any subset of contaminants (the CONTAMINANTS csv) |
## control_compounds (CONTROLS) | fbx_independent (Px_Ligase_dependent == 0 or Px_validated_WT == 0)
data.get_contaminants_and_controls(params)                         # BEFORE the load, which drops cm2rm; parts=(...) overrides the config
data.load_proteomics_data(params)                                  # DFRAW_OVERWRITE -> rebuild, else parquets; then drops EXCLUDE_DATES + cm2rm (DROP_CM2RM)
## every downstream cell must see a cohort with no fbx-independent compound, in either frame
assert not (data.df_raw['compound'].isin(data.fbx_independent).any()
            or data.MS['compound'].isin(data.fbx_independent).any()), 'fbx-independent compounds left — check CM2RM_PARTS / DROP_CM2RM'
data.compute_features(params)                                      # FEATURES_TYPE -> MF_features (+ MF morgan bits)

## OpenTargets disease scores + the big-pharma-relevant subset (see python/MS_build_ML.py:DATA)
data.load_opentargets(params)   # ot_df, cached to OT_CACHE — delete that file to force a recompute from OT_ROOT
data.filter_pharma_targets()    # ot_pharma, via the module-level PHARMA_PRIORITY_AREAS / PHARMA_DROP_AREAS

## transition shim — bare names used by the cells below
serac_df, MS, df_raw, MF, MF_features = data.serac_df, data.MS, data.df_raw, data.MF, data.MF_features
control_compounds, contaminants, cm2rm = data.control_compounds, data.contaminants, data.cm2rm
fbx_independent = data.fbx_independent
validated_targets, devalidated_targets = data.validated_targets, data.devalidated_targets
ot_df, ot_pharma = data.ot_df, data.ot_pharma
# ot_df.to_csv('data/patent/20260609_OpenTarget_list.csv', sep=',', index=False)

resolved_collections=[('AK', 931035), ('AJ', 931034)]
requested_columns=['name', 'Batch Mol-Batch ID', 'smiles', 'Px_repetition(yes/no)', 'Px_validated_WT(yes/no)', 'Px_Ligase_dependent(yes/no)', 'Px_NameLigase_dependent', 'Px_Target_info', 'Px_Target_interest', 'Largest_Vector', 'Largest_Vector_Lenght']

--- collection name=AK id=931035 ---


  AK: 25row [03:44,  8.99s/row]


  collection_rows=25

--- collection name=AJ id=931034 ---


  AJ: 13255row [03:24, 64.95row/s] 


  collection_rows=13255

total_rows=13280
multi_batch_molecules=130 (each emitted >1 row — one per batch)
> Chemical lib dim: (13194, 11) | 41 validated / 2773 devalidated targets
> 2 control + 81 contaminant compounds (from config) + 1024 fbx independent | cm2rm = contaminants + fbx_independent = 1105
> loaded cached MS (6707, 5) | df_raw (60971034, 12)
> EXCLUDE_DATES ['2026-08-13']: dropped 17,212,863 df_raw rows / 1,848 MS compounds
> DROP_CM2RM: dropped 836 compounds (8,536,861 df_raw rows) / 799 MS compounds
> checking smiles


100%|██████████| 13194/13194 [00:03<00:00, 3308.86it/s]


> FEATURES_TYPE: H237


H237 descriptors: 100%|██████████| 128/128 [00:07<00:00, 16.25it/s]


> 0 smiles failed | MF (13194, 2049) | MF_features (13139, 4470)
> loaded cached output/MS/opentargets_target_disease.parquet
> 232,004 (target, disease) rows / 11,584 targets resolved
> 78,935 / 232,004 (target, disease) rows pass the pharma filter | 9,105 unique targets
  17,071  cancer or benign tumor
   3,890  cardiovascular disease
   2,504  disorder of visual system
   3,466  endocrine system disease
   3,416  hematologic disease
   3,938  immune system disease
   3,710  infectious disease
   8,708  musculoskeletal or connective tissue disease
  25,393  nervous system disease
   9,849  nutritional or metabolic disease
   1,704  psychiatric disorder
   2,357  respiratory or thoracic disease
CPU times: user 51.8 s, sys: 31.8 s, total: 1min 23s
Wall time: 8min 15s


## 1. Single Target view - Finding under-rated compounds to enrich around

In [10]:
data.serac_df[data.serac_df['compound']=='SRB-0010624']

,compound,Batch Mol-Batch ID,smiles,Px_repetition(yes/no),Px_validated_WT(yes/no),Px_Ligase_dependent(yes/no),Px_NameLigase_dependent,Px_Target_info,Px_Target_interest,Largest_Vector,Largest_Vector_Lenght
7811,SRB-0010624,H8186546,COC(=O)C1(CS(=O)(=O)N2[C@H]3CC4=CC=C(C#N)C=C4[...,1.0,1.0,NaN,FBXO31,to_be_revised,VPS8,v2,6.0


In [8]:
'SRB-0010624' in data.fbx_independent

False

In [ ]:
data.df_raw[data.df_raw['MoleculeBatchID']=='SRB-0010624-001']

,MoleculeBatchID,MSPlate,genes,pg,logfc,pvalue,adjpval,significant,uniquecontrast,compound,batch,date
29109059,SRB-0010624-001,Pw125,NUDT4B;NUDT4,A0A024RBG1;Q9NZJ9-2,0.174736,0.006625,0.110261,0.0,SRB.0010624.001_vs_SRB.0010624.001_complement_...,SRB-0010624,001,2026-06-30
29109093,SRB-0010624-001,Pw125,SMIM26,A0A096LP01,0.022005,0.840825,0.959897,0.0,SRB.0010624.001_vs_SRB.0010624.001_complement_...,SRB-0010624,001,2026-06-30
29109127,SRB-0010624-001,Pw125,GATD3B;GATD3A,A0A0B4J2D5;P0DPI2,-0.085925,0.423798,0.793378,0.0,SRB.0010624.001_vs_SRB.0010624.001_complement_...,SRB-0010624,001,2026-06-30
29109161,SRB-0010624-001,Pw125,C20orf204,A0A1B0GTL2,0.034177,0.775215,0.940641,0.0,SRB.0010624.001_vs_SRB.0010624.001_complement_...,SRB-0010624,001,2026-06-30
29109195,SRB-0010624-001,Pw125,TTC26,A0AVF1,0.189821,0.280460,0.694699,0.0,SRB.0010624.001_vs_SRB.0010624.001_complement_...,SRB-0010624,001,2026-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...
29389559,SRB-0010624-001,Pw125,FAM169A,Q9Y6X4,-0.007579,0.849876,0.960439,0.0,SRB.0010624.001_vs_SRB.0010624.001_complement_...,SRB-0010624,001,2026-06-30
29389593,SRB-0010624-001,Pw125,ZHX2,Q9Y6X8,0.131270,0.073812,0.411700,0.0,SRB.0010624.001_vs_SRB.0010624.001_complement_...,SRB-0010624,001,2026-06-30
29389627,SRB-0010624-001,Pw125,MORC2,Q9Y6X9,0.020130,0.587511,0.865128,0.0,SRB.0010624.001_vs_SRB.0010624.001_complement_...,SRB-0010624,001,2026-06-30
29389661,SRB-0010624-001,Pw125,IVNS1ABP,Q9Y6Y0,-0.327735,0.000305,0.014168,0.0,SRB.0010624.001_vs_SRB.0010624.001_complement_...,SRB-0010624,001,2026-06-30


In [11]:
## get hot target list: row 1 is a title banner, so row 2 carries the column names
hot_df = pd.read_excel(params.dbx(params.HOT_TARGETS), sheet_name='Master Overview', header=1)
print(f'> hot targets: {hot_df.shape[0]:,} rows x {hot_df.shape[1]} cols')

## get SAR predictability per gene
gene_metrics = pd.read_csv(GENE_SAR_OUT)

## match the hot target names onto the genes our screen measured (fn.GENE_ALIASES covers protein names)
hot_map = fn.match_gene_symbols(hot_df['Target'].dropna(), gene_metrics['gene'])
hot_r2  = (hot_map[hot_map['matched']].merge(gene_metrics, left_on='symbol', right_on='gene')
                                      .sort_values('R2', ascending=False))
print(f"> matched {hot_map['matched'].sum()}/{len(hot_map)} targets")
print(f"> never measured: {sorted(hot_map.loc[~hot_map['matched'], 'target'])}")

## + compounds per activity class that significantly down-regulate it, and the canonical MS score
## (max of -logfc * -log10(p) over significant rows — Px_interface.py:127-131 / :386)
hot_tbl = (hot_r2.merge(fn.count_hits_by_activity(data.df_raw, data.MS, genes=hot_r2['symbol']),
                        left_on='symbol', right_on='genes').drop(columns='genes')
                 .merge(fn.gene_ms_score(data.df_raw, genes=hot_r2['symbol']),
                        left_on='symbol', right_index=True, how='left'))
hot_tbl[['target', 'symbol', 'R2', 'n', 'spearman_r', 'ms_score', 'Single (1)', 'Low (2-10)',
         'Medium (11-25)', 'High (>25)', 'hits']].round(2).head(20)

> hot targets: 69 rows x 24 cols
> matched 56/69 targets
> never measured: ['AR', 'ER (ESR1)', 'ERG', 'FAP', 'IKZF2 (Helios)', 'IRF4', 'MAGL', 'MYCN', 'NLRP3', 'PTPN22', 'TAL1', 'TOX', 'TRIB2']


,target,symbol,R2,n,spearman_r,ms_score,Single (1),Low (2-10),Medium (11-25),High (>25),hits
0,PCSK9,PCSK9,0.34,4290,0.28,100.00,0,1,15,83,99
1,KEAP1,KEAP1,0.32,4290,0.42,21.59,0,0,0,3,3
2,APOL1,APOL1,0.26,4290,0.21,100.00,0,0,1,79,80
3,FOXA1,FOXA1,0.24,4290,0.42,100.00,0,0,0,7,7
4,SMARCA2 (BRM),SMARCA2,0.19,4290,0.39,NaN,0,0,0,0,0
5,Menin (MEN1),MEN1,0.14,4290,0.24,NaN,0,0,0,0,0
6,BLM (Bloom helicase),BLM,0.12,4290,0.14,28.17,0,0,0,1,1
7,TEAD1,TEAD1,0.10,4290,0.32,NaN,0,0,0,0,0
8,PTPN2 (TC-PTP),PTPN2,0.10,4290,0.09,NaN,0,0,0,0,0
9,RECQL4,RECQL4,0.10,3913,0.19,14.28,0,0,0,8,8


In [ ]:
## Looking for gene specific under-appreciated compounds — set GENE to any measured target.
## Pareto gate (layer) + isolation (d_any), over the volcanoes where GENE is NOT significant.
GENE     = 'SMARCA2'   # any symbol in data.df_raw['genes'], e.g. PCSK9 / APOL1 / KEAP1 / FOXA1
PARTNERS = []        # co-regulated genes to control for (PCSK9: the sterol block); [] skips it

_g = data.df_raw[data.df_raw['genes'] == GENE]
if _g.empty:
    raise ValueError(f'{GENE} is not measured — no rows in df_raw')
print(f'> {GENE}: {_g["uniquecontrast"].nunique():,} experiments | '
      f'{int(((_g["significant"] == 1) & (_g["logfc"] < 0)).sum()):,} significant-down')

standout = fn.rank_gene_standout(data.df_raw, GENE, max_layer=params.STANDOUT_MAX_LAYER,
                                 radius=params.STANDOUT_RADIUS, scale_pct=params.STANDOUT_SCALE_PCT)
_meta = data.df_raw.drop_duplicates('uniquecontrast').set_index('uniquecontrast')
standout['compound'] = standout['uniquecontrast'].map(_meta['compound'])
standout['plate'] = standout['uniquecontrast'].map(_meta['MSPlate'])   # NaN for the 20260429 CDD export
standout = (standout[~standout['compound'].isin(set(data.cm2rm))]
            .merge(data.MS[['compound', 'ndown', 'activity']], on='compound', how='left'))

## control: a true standout moves GENE alone, so its co-regulated partners must stay flat
standout[['partners_down', 'partners_logfc']] = np.nan
if PARTNERS:
    _p = (data.df_raw[data.df_raw['genes'].isin(PARTNERS)
                      & data.df_raw['uniquecontrast'].isin(standout['uniquecontrast'])]
          .assign(_down=lambda z: (z['significant'] == 1) & (z['logfc'] < 0)))
    standout[['partners_down', 'partners_logfc']] = (
        _p.groupby('uniquecontrast').agg(partners_down=('_down', 'sum'),
                                         partners_logfc=('logfc', 'mean'))
          .reindex(standout['uniquecontrast']).to_numpy())

print(f'> {len(standout)} non-significant volcanoes within Pareto layer {params.STANDOUT_MAX_LAYER}'
      + (f' | {int((standout["partners_down"] == 0).sum())} with no partner moving' if PARTNERS else ''))

standout = standout[standout['activity']!='High (>25)']

standout[['compound', 'plate', 'logfc', 'pvalue', 'y', 'n_dom', 'layer', 'd_any', 'd_bulk',
          'density', 'n_frontier', 'ndown', 'activity', 'partners_down', 'partners_logfc']]

In [ ]:
## plot volcano of selectd compound:
## interactive: fill = significance + direction (grey / SILENT_C down / ACTIVE_C up), dark rings + labels
## = Pareto layers 1..STANDOUT_MAX_LAYER (deeper = fainter), blue ring = GENE, structure top-right.
## gold rings = the STRING first shell of GENE; only the symbol leaves the machine, the match is local
## the buttons and the slider move through the standout ranks — no need to edit a value
_shell = fn.string_first_shell(GENE, species=params.STRING_SPECIES, min_score=params.STRING_MIN_SCORE,
                               limit=params.STRING_MAX_PARTNERS, cache_dir=params.STRING_CACHE)

_out  = w.Output()
_rank = w.IntSlider(value=0, min=0, max=len(standout) - 1, description='rank',
                    continuous_update=False, layout=w.Layout(width='420px'))
_prev, _next = w.Button(description='< prev'), w.Button(description='next >')

def _show_rank(rank):
    r = standout.iloc[rank]
    with _out:
        clear_output(wait=True)
        print(f"> rank {rank}/{len(standout) - 1}: {r['compound']} {r['plate']} | {GENE} logfc "
              f"{r['logfc']:.3f} p {r['pvalue']:.2e} | layer {int(r['layer'])} "
              f"| d_any {r['d_any']:.3f} | ndown {r['ndown']:.0f}")
        fn.plot_volcano_pareto(data.df_raw, r['uniquecontrast'], highlight=GENE,
                               max_layer=params.STANDOUT_MAX_LAYER, up_color=params.ACTIVE_C,
                               down_color=params.SILENT_C, compound=r['compound'],
                               partners=_shell['partner'].tolist(),
                               struct_dir=params.SRB_PNG_DIR,   # None
                               struct_size=0.45).show()

def _step(delta):
    _rank.value = min(max(_rank.value + delta, _rank.min), _rank.max)

_prev.on_click(lambda _: _step(-1))
_next.on_click(lambda _: _step(+1))
_rank.observe(lambda c: _show_rank(c['new']), names='value')
display(w.HBox([_prev, _next, _rank]), _out)
_show_rank(_rank.value)

## 2. Systematic multi-target approach

In [12]:
## every hot target gets its own HTML report — skip a gene with no standout or only High (>25)
## gold rings = that target's STRING first shell; only the symbol leaves the machine
REPORT_DIR = params.DROPBOX_ML + params.STANDOUT_REPORT_DBX
TARGETS    = hot_tbl['symbol'].dropna().unique()
Path(params.STANDOUT_REPORT_LOC).mkdir(parents=True, exist_ok=True)
rows, skipped = [], {}

for _gene in tqdm(TARGETS, desc='targets'):
    _s   = fn.standout_table(data.df_raw, data.MS, _gene, cm2rm=data.cm2rm,
                             max_layer=params.STANDOUT_MAX_LAYER, radius=params.STANDOUT_RADIUS,
                             scale_pct=params.STANDOUT_SCALE_PCT)
    _sel = _s[_s['activity'] != 'High (>25)'].reset_index(drop=True)
    if not len(_sel):
        skipped[_gene] = 'no standout' if not len(_s) else 'only High (>25)'
        continue
    _shell = fn.string_first_shell(_gene, species=params.STRING_SPECIES,
                                   min_score=params.STRING_MIN_SCORE,
                                   limit=params.STRING_MAX_PARTNERS, cache_dir=params.STRING_CACHE)
    _name = f'{date.today():%Y%m%d}_{_gene}_report.html'
    _loc  = os.path.join(params.STANDOUT_REPORT_LOC, _name)
    Path(_loc).write_text(fn.standout_report_html(
        data.df_raw, _gene, _sel, max_layer=params.STANDOUT_MAX_LAYER, up_color=params.ACTIVE_C,
        down_color=params.SILENT_C, partners=_shell['partner'].tolist(),
        struct_dir=params.SRB_PNG_DIR, max_volcanoes=params.STANDOUT_REPORT_MAX,
        note=f'Gold rings = the {len(_shell)} STRING first-shell partners.'), encoding='utf-8')
    fn.push_rclone(_loc, REPORT_DIR + _name)
    rows.append({'gene': _gene, 'candidates': len(_sel), 'top_d_any': _sel['d_any'].iloc[0],
                 'best_layer': int(_sel['layer'].min()), 'MB': os.path.getsize(_loc) / 1e6,
                 'report': _name})

reports = pd.DataFrame(rows).sort_values('top_d_any', ascending=False).reset_index(drop=True)
print(f'> {len(reports)} reports -> {REPORT_DIR} ({reports["MB"].sum():.0f} MB total) '
      f'| {len(skipped)} genes skipped')
print('> skipped:', skipped)
reports.round(3)

targets:   0%|          | 0/56 [00:00<?, ?it/s]

> STRING PCSK9: 28 partners from cache (data/external/string/PCSK9_9606_700_50.tsv)
> STRING APOL1: 35 partners from cache (data/external/string/APOL1_9606_700_50.tsv)
> STRING SMARCA2: 50 partners from cache (data/external/string/SMARCA2_9606_700_50.tsv)
> STRING MEN1: 50 partners from cache (data/external/string/MEN1_9606_700_50.tsv)
> STRING RECQL4: 50 partners from cache (data/external/string/RECQL4_9606_700_50.tsv)
> STRING HIF1A: 50 partners from cache (data/external/string/HIF1A_9606_700_50.tsv)
> STRING EXO1: 50 partners from cache (data/external/string/EXO1_9606_700_50.tsv)
> STRING HTT: 50 partners from cache (data/external/string/HTT_9606_700_50.tsv)
> STRING WEE1: 40 partners from cache (data/external/string/WEE1_9606_700_50.tsv)
> STRING BCL6: 50 partners from cache (data/external/string/BCL6_9606_700_50.tsv)
> STRING EPAS1: 50 partners from cache (data/external/string/EPAS1_9606_700_50.tsv)
> STRING TYK2: 50 partners from cache (data/external/string/TYK2_9606_700_50.tsv)


,gene,candidates,top_d_any,best_layer,MB,report
0,STAT6,22,4.477,1,5.351,20260929_STAT6_report.html
1,RIPK1,4,3.367,1,2.293,20260929_RIPK1_report.html
2,APOL1,11,2.935,1,5.321,20260929_APOL1_report.html
3,SOX4,11,2.402,1,5.296,20260929_SOX4_report.html
4,CCNE1,3,1.658,2,1.707,20260929_CCNE1_report.html
5,PCSK9,24,1.641,1,5.302,20260929_PCSK9_report.html
6,HIF1A,30,1.617,1,5.271,20260929_HIF1A_report.html
7,JAK2,6,0.994,1,3.160,20260929_JAK2_report.html
8,SLC1A5,4,0.806,1,2.090,20260929_SLC1A5_report.html
9,MYC,5,0.789,1,2.624,20260929_MYC_report.html
